In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

In [0]:

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

In [0]:
columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

In [0]:
df_bronze = spark.createDataFrame(data, columns)

df_bronze.write.format("delta") \
.mode("append") \
.save("/Volumes/workspace/default/volume_flipkart")

df_bronze.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_bronze = df_bronze.withColumn("amount", col("amount").cast("int")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("quantity", col("quantity").cast("int"))


In [0]:
from pyspark.sql.functions import col

df_bronze_clean = df_bronze \
    .fillna({"city": "Unknown"}) \
    .filter(col("amount") != 0)

df_bronze_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_bronze_clean = df_bronze_clean.dropDuplicates()

In [0]:
df_bronze_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
windowSpec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_bronze_clean = df_bronze_clean \
    .withColumn("rn", row_number().over(windowSpec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

In [0]:
df_bronze_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_bronze_clean = df_bronze_clean.filter(col("amount") > 0)

In [0]:
df_bronze_clean.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_silver = df_bronze_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/volume_flipkart")

In [0]:
df_silver_display = spark.read.format("delta").load("/Volumes/workspace/default/volume_flipkart")
df_silver_display.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.functions import sum

sales_by_product = df_silver_display.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

sales_by_product.display()

product,total_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


In [0]:
sales_by_category = df_silver_display.groupBy("category") \
    .agg(sum("amount").alias("total_sales"))

sales_by_category.display()

category,total_sales
Electronics,160000
Accessories,11000


In [0]:
sales_by_city = df_silver_display.groupBy("city") \
    .agg(sum("amount").alias("total_sales"))

sales_by_city.display()

city,total_sales
Delhi,55000
Chennai,33000
Unknown,3000
Hyderabad,72000
Mumbai,8000


In [0]:
from pyspark.sql.functions import count

orders_per_customer = df_silver_display.groupBy("customer_id") \
    .agg(count("order_id").alias("total_orders"))

orders_per_customer.display()

customer_id,total_orders
C005,1
C004,1
C003,1
C001,2
C002,1
C007,1


In [0]:
customer_spend = df_silver_display.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spending"))

customer_spend.display()

customer_id,total_spending
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


In [0]:
top_product = sales_by_product.orderBy(col("total_sales").desc())

top_product.display()

product,total_sales
Laptop,105000
Mobile,33000
Tablet,22000
Watch,8000
Headphones,3000


In [0]:
top_customer = customer_spend.orderBy(col("total_spending").desc())

top_customer.display()

customer_id,total_spending
C001,72000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000
